In [1]:
import pandas as pd
import pickle
import warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import RFE
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

In [2]:
dataset=pd.read_csv("house_price_cleaned.csv")
dataset

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450.0,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500.0
1,2,20,RL,80.0,9600.0,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500.0
2,3,60,RL,68.0,11250.0,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500.0
3,4,70,RL,60.0,9550.0,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000.0
4,5,60,RL,84.0,14260.0,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,1456,60,RL,62.0,7917.0,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2007,WD,Normal,175000.0
1456,1457,20,RL,85.0,13175.0,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,2,2010,WD,Normal,210000.0
1457,1458,70,RL,66.0,9042.0,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,GdPrv,Shed,0,5,2010,WD,Normal,266500.0
1458,1459,20,RL,68.0,9717.0,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,142125.0


In [3]:
dataset.shape

(1460, 81)

In [4]:
dataset=pd.get_dummies(dataset,dtype=int,drop_first=True)
dataset

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,SaleType_ConLI,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
0,1,60,65.0,8450.0,7,5.0,2003,2003,196.0,706.0,...,0,0,0,0,1,0,0,0,1,0
1,2,20,80.0,9600.0,6,7.5,1976,1976,0.0,978.0,...,0,0,0,0,1,0,0,0,1,0
2,3,60,68.0,11250.0,7,5.0,2001,2002,162.0,486.0,...,0,0,0,0,1,0,0,0,1,0
3,4,70,60.0,9550.0,7,5.0,1915,1970,0.0,216.0,...,0,0,0,0,1,0,0,0,0,0
4,5,60,84.0,14260.0,8,5.0,2000,2000,350.0,655.0,...,0,0,0,0,1,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,1456,60,62.0,7917.0,6,5.0,1999,2000,0.0,0.0,...,0,0,0,0,1,0,0,0,1,0
1456,1457,20,85.0,13175.0,6,6.0,1978,1988,119.0,790.0,...,0,0,0,0,1,0,0,0,1,0
1457,1458,70,66.0,9042.0,7,7.5,1941,2006,0.0,275.0,...,0,0,0,0,1,0,0,0,1,0
1458,1459,20,68.0,9717.0,5,6.0,1950,1996,0.0,49.0,...,0,0,0,0,1,0,0,0,1,0


In [5]:
dataset.isnull().sum()

Id                       0
MSSubClass               0
LotFrontage              0
LotArea                  0
OverallQual              0
                        ..
SaleCondition_AdjLand    0
SaleCondition_Alloca     0
SaleCondition_Family     0
SaleCondition_Normal     0
SaleCondition_Partial    0
Length: 246, dtype: int64

In [6]:
indep_X = dataset.drop(["Id","SalePrice"], axis=1)
dep_Y = dataset["SalePrice"]

In [7]:
X_train, X_test, y_train, y_test = (train_test_split(indep_X, dep_Y, test_size=0.30, random_state=0))

In [8]:
n = 7
rfe = RFE(estimator=RandomForestRegressor(random_state=0), n_features_to_select=n)
X_train = rfe.fit_transform(X_train, y_train)
X_test = rfe.transform(X_test)

In [9]:
pipeline = Pipeline ([("rf",RandomForestRegressor(random_state=0))])

In [10]:
param_grid = {'rf__n_estimators': [50, 100, 200],'rf__max_depth': [None, 10, 20]}
search = GridSearchCV(pipeline, param_grid, cv=3, n_jobs=-1)
search.fit(X_train, y_train)
best_model = search.best_estimator_

In [11]:
r2 = best_model.score(X_test,y_test)

print("Best Random Forest Model:", best_model)
print("R² Score:", r2)

Best Random Forest Model: Pipeline(steps=[('rf',
                 RandomForestRegressor(max_depth=20, n_estimators=200,
                                       random_state=0))])
R² Score: 0.8686869391298007


In [12]:
final_pipeline = Pipeline([("rfe",RFE(estimator=RandomForestRegressor(random_state=0),n_features_to_select=n)),
                           ("rf",RandomForestRegressor(n_estimators=search.best_params_['rf__n_estimators'],
                                                       max_depth=search.best_params_['rf__max_depth'],
                                                       random_state=0))])
final_pipeline.fit(indep_X,dep_Y)

with open("finalized_model_RFE_Residential_House_Price_data.sav","wb") as f:
    pickle.dump(final_pipeline,f)

In [13]:
loaded_model=pickle.load(open("finalized_model_RFE_Residential_House_Price_data.sav","rb"))

#### Deployment model

In [15]:
selected_columns = indep_X.columns[rfe.get_support()]

print("Selected Features")
print(selected_columns)

Selected Features
Index(['OverallQual', 'YearBuilt', 'BsmtFinSF1', 'TotalBsmtSF', 'GrLivArea',
       'GarageCars', 'GarageArea'],
      dtype='object')


In [16]:
X_selected = indep_X[selected_columns]

# Train a model only on the selected features
deploy_model = RandomForestRegressor(
    n_estimators=search.best_params_['rf__n_estimators'],
    max_depth=search.best_params_['rf__max_depth'],
    random_state=0
)

deploy_model.fit(X_selected, dep_Y)

RandomForestRegressor(max_depth=20, n_estimators=200, random_state=0)

In [17]:
import pickle

with open("house_price_deploy_model.sav", "wb") as f:
    pickle.dump(deploy_model, f)

In [18]:
loaded_model = pickle.load(open("house_price_deploy_model.sav", "rb"))

sample = indep_X.iloc[[0]][selected_columns]

prediction = loaded_model.predict(sample)

print("Actual Price:", dep_Y.iloc[0])
print("Predicted Price:", prediction[0])

Actual Price: 208500.0
Predicted Price: 206412.51324154134
